# Offline RL: CQL / IQL / TD3+BC — Full Training on T4 GPU

**Pipeline:**
1. Clone repo from GitHub
2. Install dependencies + D4RL real datasets (Minari)
3. Train all 3 agents on `hopper-medium-v2`
4. Plot learning curves + Q-overestimation analysis
5. Push results, plots, and checkpoints back to GitHub

> Make sure **Runtime → Change runtime type → T4 GPU** is selected before running.

## 0. GPU Check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

## 1. Clone Your Repo

In [ ]:
# ── EDIT THIS ──────────────────────────────────────────────────────────────
GITHUB_USERNAME = "your-github-username"   # e.g. "john-doe"
GITHUB_TOKEN    = "ghp_xxxxxxxxxxxx"       # Personal Access Token (repo scope)
                                            # Get at: github.com/settings/tokens
REPO_NAME       = "offline-rl-conservative-q-learning"
GIT_EMAIL       = "your@email.com"
GIT_NAME        = "Your Name"
# ───────────────────────────────────────────────────────────────────────────

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

import os
os.chdir('/content')
!git clone {REPO_URL}
os.chdir(f'/content/{REPO_NAME}')

!git config user.email "{GIT_EMAIL}"
!git config user.name  "{GIT_NAME}"

print('Repo cloned. Working directory:', os.getcwd())
!ls

## 2. Install Dependencies

In [ ]:
# Core deps — PyTorch is already installed on Colab
!pip install -q minari gymnasium[mujoco] rich tqdm einops

# Minari D4RL-compatible datasets (no MuJoCo license needed)
# These are the standard offline RL benchmarks:
#   hopper-medium-v2: medium-quality hopper locomotion data
#   halfcheetah-medium-v2: medium-quality half-cheetah data
print('Dependencies installed.')

## 3. Download Real D4RL Datasets via Minari

In [ ]:
import minari

# The MuJoCo locomotion datasets in Minari use this naming format:
#   mujoco/hopper/medium-v2   (NOT mujoco/hopper/medium-v2)
# These are the D4RL-equivalent datasets re-hosted by the Farama Foundation.
#
# 'medium-v2' = data from a policy trained to ~1/3 of expert performance
# This is the standard offline RL benchmark from Fu et al. (2020)

DATASETS = [
    "mujoco/hopper/medium-v2",
    "mujoco/halfcheetah/medium-v2",
]

for ds in DATASETS:
    try:
        minari.load_dataset(ds, download=False)
        print(f'Already cached: {ds}')
    except FileNotFoundError:
        print(f'Downloading: {ds} (~50MB) ...')
        minari.download_dataset(ds)
        print(f'Done: {ds}')

# Verify
for ds in DATASETS:
    d = minari.load_dataset(ds)
    print(f'{ds}: {d.total_steps:,} transitions, '
          f'obs={d.observation_space.shape}, act={d.action_space.shape}')

print('\nAll datasets ready.')

## 4. Quick Sanity Check — Dataset Exploration

In [ ]:
import sys
sys.path.insert(0, '/content/' + REPO_NAME)

import numpy as np
import matplotlib.pyplot as plt
from src.data.dataset import load_dataset
from src.data.replay_buffer import OfflineReplayBuffer

dataset, env_info = load_dataset("mujoco/hopper/medium-v2")

print(f"obs_dim  : {env_info['obs_dim']}")
print(f"act_dim  : {env_info['act_dim']}")
print(f"n_trans  : {len(dataset['rewards']):,}")
print(f"reward   : [{dataset['rewards'].min():.2f}, {dataset['rewards'].max():.2f}]")

# Plot reward distribution — important for understanding data quality
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Dataset Exploration: Hopper Medium v2', fontsize=13, fontweight='bold')

axes[0].hist(dataset['rewards'], bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_title('Reward Distribution')
axes[0].set_xlabel('Reward per step')

axes[1].hist(dataset['actions'].flatten(), bins=60, color='coral', edgecolor='white', linewidth=0.3)
axes[1].set_title('Action Distribution\n(behavior policy)')
axes[1].set_xlabel('Action value')

# Episode return distribution
terminals = np.where(dataset['terminals'])[0]
starts = np.concatenate([[0], terminals[:-1]+1])
ep_returns = [dataset['rewards'][s:e].sum() for s, e in zip(starts, terminals)]
axes[2].hist(ep_returns, bins=30, color='mediumseagreen', edgecolor='white', linewidth=0.3)
axes[2].set_title('Episode Return Distribution')
axes[2].set_xlabel('Episode return')

plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/dataset_exploration.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved.')

## 5. Training — All 3 Agents

We train each agent for **500k steps** on `hopper-medium-v2`.  
Expected time on T4: ~8 min per agent (~25 min total).

| Agent | Core idea | Expected score (Hopper-medium) |
|---|---|---|
| TD3+BC | BC regularization on actor | ~59 |
| CQL | Conservative Q lower-bound | ~79 |
| IQL | Expectile regression, no OOD queries | ~75 |

In [ ]:
import subprocess, json, time

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
DATASET  = "mujoco/hopper/medium-v2"
STEPS    = 500_000
BATCH    = 1024          # larger batch = faster on GPU
EVAL_FREQ= 10_000
EVAL_EPS = 10

os.makedirs('results/logs', exist_ok=True)

agents = [
    ("td3bc", {"bc_alpha": 2.5}),
    ("cql",   {"cql_alpha": 5.0, "with_lagrange": False}),
    ("iql",   {"expectile": 0.7, "temperature": 3.0}),
]

results = {}

for agent_name, extra_args in agents:
    print(f'\n{"="*60}')
    print(f'  Training {agent_name.upper()} on {DATASET}')
    print(f'{"="*60}')
    t0 = time.time()

    cmd = [
        "python", "train.py",
        "--agent",    agent_name,
        "--dataset",  DATASET,
        "--steps",    str(STEPS),
        "--batch",    str(BATCH),
        "--eval_freq",str(EVAL_FREQ),
        "--eval_eps", str(EVAL_EPS),
        "--device",   DEVICE,
        "--save_dir", "results/logs",
    ]
    for k, v in extra_args.items():
        if isinstance(v, bool):
            if v: cmd.append(f"--{k}")
        else:
            cmd += [f"--{k}", str(v)]

    proc = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = (time.time() - t0) / 60

    if proc.returncode != 0:
        print('STDERR:', proc.stderr[-2000:])
    else:
        print(proc.stdout[-1500:])
        print(f'Finished in {elapsed:.1f} min')

    results[agent_name] = {"elapsed_min": elapsed, "returncode": proc.returncode}

print('\nAll agents trained.')

## 6. Parse Logs & Plot Results

In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.ndimage import uniform_filter1d

COLORS = {"td3bc": "#E8734A", "cql": "#4A90D9", "iql": "#5BB882"}
LABELS = {"td3bc": "TD3+BC", "cql": "CQL", "iql": "IQL"}

def load_eval_csv(agent_name):
    pattern = f"results/logs/{agent_name}_*"
    dirs = sorted(glob.glob(pattern))
    if not dirs:
        return None
    latest = dirs[-1]
    csv_path = os.path.join(latest, 'eval_metrics.csv')
    return pd.read_csv(csv_path) if os.path.exists(csv_path) else None

def load_train_csv(agent_name):
    pattern = f"results/logs/{agent_name}_*"
    dirs = sorted(glob.glob(pattern))
    if not dirs:
        return None
    latest = dirs[-1]
    csv_path = os.path.join(latest, 'train_metrics.csv')
    return pd.read_csv(csv_path) if os.path.exists(csv_path) else None

# ── Figure 1: Normalized Score Curves ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Offline RL on Hopper-Medium-v2 — Normalized Score', fontsize=14, fontweight='bold')

all_evals = {}
for agent_name in ["td3bc", "cql", "iql"]:
    df = load_eval_csv(agent_name)
    if df is None:
        print(f'No eval log found for {agent_name}')
        continue
    all_evals[agent_name] = df

# Subplot 1: all agents on same axes
ax = axes[0]
for agent_name, df in all_evals.items():
    ax.plot(df['step']/1e6, df['normalized_score'],
            color=COLORS[agent_name], label=LABELS[agent_name], alpha=0.9, linewidth=2)
ax.axhline(y=100, color='gray', linestyle='--', linewidth=0.8, label='Expert (100)')
ax.axhline(y=0,   color='gray', linestyle=':',  linewidth=0.8, label='Random (0)')
ax.set_xlabel('Steps (M)'); ax.set_ylabel('Normalized Score')
ax.set_title('All Agents')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Subplot 2: Q-value overestimation (train logs)
ax = axes[1]
for agent_name in ["td3bc", "cql", "iql"]:
    df = load_train_csv(agent_name)
    if df is None: continue
    q_col = next((c for c in ['q1_mean','q1_data_mean'] if c in df.columns), None)
    if q_col is None: continue
    smoothed = uniform_filter1d(df[q_col].values, size=500)
    ax.plot(df['step']/1e6, smoothed,
            color=COLORS[agent_name], label=LABELS[agent_name], linewidth=2)
ax.set_xlabel('Steps (M)'); ax.set_ylabel('Mean Q-value')
ax.set_title('Q-value Evolution\n(CQL stays conservative)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# Subplot 3: CQL penalty over time
ax = axes[2]
df_cql = load_train_csv("cql")
if df_cql is not None and 'cql_penalty' in df_cql.columns:
    smoothed = uniform_filter1d(df_cql['cql_penalty'].values, size=500)
    ax.plot(df_cql['step']/1e6, smoothed,
            color=COLORS['cql'], linewidth=2, label='CQL penalty')
    ax.set_xlabel('Steps (M)'); ax.set_ylabel('CQL Penalty (logsumexp − E_data[Q])')
    ax.set_title('CQL Conservative Penalty\n(should converge to ~0)')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
else:
    ax.text(0.5, 0.5, 'CQL log not found', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('results/plots/learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Learning curves saved.')

In [ ]:
# ── Figure 2: Final Score Comparison Bar Chart ─────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
fig.suptitle('Final Normalized Score — Hopper-Medium-v2', fontsize=13, fontweight='bold')

final_scores = {}
for agent_name, df in all_evals.items():
    # Use mean of last 5 eval points for stability
    final_scores[agent_name] = df['normalized_score'].tail(5).mean()

bars = ax.bar(
    [LABELS[a] for a in final_scores],
    list(final_scores.values()),
    color=[COLORS[a] for a in final_scores],
    width=0.5, edgecolor='white', linewidth=1.2
)
for bar, (agent, score) in zip(bars, final_scores.items()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{score:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=12)

# Paper-reported baselines (from original papers)
paper_scores = {"TD3+BC": 59.3, "CQL": 79.4, "IQL": 75.1}
for i, (label, paper_score) in enumerate(paper_scores.items()):
    ax.scatter(i, paper_score, marker='_', s=500, color='black', zorder=5, linewidths=2)
    ax.text(i + 0.28, paper_score, f'paper: {paper_score}', va='center', fontsize=8, color='#555')

ax.axhline(100, color='gray', linestyle='--', linewidth=0.8, label='Expert')
ax.axhline(0,   color='gray', linestyle=':',  linewidth=0.8, label='Random')
ax.set_ylabel('D4RL Normalized Score')
ax.set_ylim(-5, 115)
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/plots/final_scores.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary table
print('\n── Final Results ──────────────────────────')
print(f'{"Agent":<10} {"Our Score":>12} {"Paper Score":>12}')
print('-'*36)
for agent, score in final_scores.items():
    paper = paper_scores.get(LABELS[agent], '-')
    print(f'{LABELS[agent]:<10} {score:>12.1f} {str(paper):>12}')
print('─'*36)

## 7. IQL Expectile Analysis
Visualize how different `expectile` values change what V(s) approximates.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# Demonstrate expectile regression theory
# For a distribution of Q-values, show what tau controls
np.random.seed(0)
Q_samples = np.concatenate([
    np.random.normal(20, 5, 300),   # most actions: mediocre
    np.random.normal(60, 10, 80),   # some good actions
    np.random.normal(90, 5, 20),    # rare excellent actions
])

def fit_expectile(Q, tau, n_iter=2000, lr=0.1):
    """Fit V to minimize expectile loss against Q samples."""
    V = np.mean(Q)  # start at mean
    for _ in range(n_iter):
        u = Q - V
        w = np.where(u > 0, tau, 1 - tau)
        grad = -2 * np.mean(w * u)  # d/dV of expectile loss
        V -= lr * grad
    return V

taus = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]
V_values = [fit_expectile(Q_samples, tau) for tau in taus]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('IQL: Expectile Regression Theory', fontsize=13, fontweight='bold')

ax = axes[0]
ax.hist(Q_samples, bins=50, density=True, color='lightsteelblue', edgecolor='white', label='Q distribution')
cmap = plt.cm.plasma
for i, (tau, V) in enumerate(zip(taus, V_values)):
    color = cmap(i / len(taus))
    ax.axvline(V, color=color, linewidth=2, label=f'τ={tau} → V={V:.1f}')
ax.axvline(np.max(Q_samples), color='red', linestyle='--', linewidth=1.5, label=f'max Q={np.max(Q_samples):.1f}')
ax.legend(fontsize=7.5, loc='upper left')
ax.set_xlabel('Q-value'); ax.set_ylabel('Density')
ax.set_title('V(s) converges to max Q\nas τ → 1')

ax = axes[1]
ax.plot(taus, V_values, 'o-', color='#4A90D9', linewidth=2, markersize=8)
ax.axhline(np.max(Q_samples), color='red', linestyle='--', linewidth=1.5, label='max Q')
ax.axhline(np.mean(Q_samples), color='gray', linestyle=':', linewidth=1.5, label='mean Q (τ=0.5)')
ax.set_xlabel('Expectile τ'); ax.set_ylabel('Fitted V value')
ax.set_title('V(s) vs Expectile τ\n(τ=0.7 used in IQL paper for locomotion)')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/plots/expectile_theory.png', dpi=150, bbox_inches='tight')
plt.show()
print('Expectile analysis saved.')

## 8. CQL — OOD Q-value Analysis
Show that CQL keeps Q conservative while vanilla Q-learning overestimates.

In [ ]:
# Load trained CQL agent and probe Q-values on in-distribution vs OOD actions
import sys
sys.path.insert(0, '/content/' + REPO_NAME)

import glob, torch
from src.agents.cql import CQL
from src.data.dataset import load_dataset
from src.data.replay_buffer import OfflineReplayBuffer

dataset, env_info = load_dataset("mujoco/hopper/medium-v2")
buffer = OfflineReplayBuffer(env_info['obs_dim'], env_info['act_dim'],
                              normalize_obs=True, normalize_reward=True)
buffer.load_from_dict(dataset)

# Load best CQL checkpoint
cql_dirs = sorted(glob.glob('results/logs/cql_*'))
if cql_dirs:
    ckpt_path = os.path.join(cql_dirs[-1], 'best.pt')
    agent = CQL(env_info['obs_dim'], env_info['act_dim'], device='cpu')
    agent.load(ckpt_path)

    batch = buffer.sample(512)
    S = batch.observations
    A_data = batch.actions  # in-distribution actions

    # OOD actions: uniform random
    A_ood = torch.FloatTensor(512, env_info['act_dim']).uniform_(-1, 1)

    with torch.no_grad():
        q1_data, _ = agent.critic(S, A_data)
        q1_ood,  _ = agent.critic(S, A_ood)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('CQL: Q-values In-Distribution vs OOD Actions', fontsize=13, fontweight='bold')

    axes[0].hist(q1_data.numpy().flatten(), bins=40, color='#5BB882', alpha=0.8,
                 label=f'Dataset actions (mean={q1_data.mean():.1f})', edgecolor='white')
    axes[0].hist(q1_ood.numpy().flatten(), bins=40, color='#E8734A', alpha=0.8,
                 label=f'Random OOD actions (mean={q1_ood.mean():.1f})', edgecolor='white')
    axes[0].set_xlabel('Q-value'); axes[0].set_ylabel('Count')
    axes[0].set_title('CQL keeps Q(OOD) < Q(data)\n— the conservative guarantee')
    axes[0].legend(fontsize=9)

    # Scatter: Q(data) vs Q(ood) for same state
    axes[1].scatter(q1_data.numpy().flatten()[:200],
                    q1_ood.numpy().flatten()[:200],
                    alpha=0.4, s=20, color='#4A90D9')
    lims = [min(q1_data.min().item(), q1_ood.min().item()),
            max(q1_data.max().item(), q1_ood.max().item())]
    axes[1].plot(lims, lims, 'r--', linewidth=1.5, label='Q(data)=Q(OOD) line')
    axes[1].set_xlabel('Q(s, a_data)'); axes[1].set_ylabel('Q(s, a_ood)')
    axes[1].set_title('Most points below diagonal:\nCQL penalizes OOD Q-values')
    axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('results/plots/cql_ood_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('OOD analysis saved.')
else:
    print('No CQL checkpoint found — run training first.')

## 9. Push Everything Back to GitHub

In [ ]:
import subprocess, datetime

os.chdir(f'/content/{REPO_NAME}')

# Create results/README.md summarizing the run
os.makedirs('results', exist_ok=True)
with open('results/README.md', 'w') as f:
    f.write(f"""# Training Results — Hopper-Medium-v2

Trained on Google Colab T4 GPU.  
Date: {datetime.datetime.now().strftime('%Y-%m-%d')}  
Steps: {STEPS:,} | Batch: {BATCH} | Device: {DEVICE}

## Final Normalized Scores

| Agent | Our Score | Paper Score |
|---|---|---|
""")
    for agent, score in final_scores.items():
        paper = paper_scores.get(LABELS[agent], '-')
        f.write(f"| {LABELS[agent]} | {score:.1f} | {paper} |\n")
    f.write("""
## Plots
- `plots/dataset_exploration.png` — reward/action/return distributions
- `plots/learning_curves.png`    — normalized score, Q-values, CQL penalty
- `plots/final_scores.png`       — bar chart vs paper baselines
- `plots/expectile_theory.png`   — IQL expectile regression visualization
- `plots/cql_ood_analysis.png`   — CQL conservative Q-value analysis
""")

# Stage all results
!git add results/
!git add experiments/

# Commit
commit_msg = f"Add training results: CQL/IQL/TD3BC on hopper-medium-v2 ({STEPS//1000}k steps, T4 GPU)"
!git commit -m "{commit_msg}"

# Push
!git push origin main

print('\nAll results pushed to GitHub.')
print(f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}')

## 10. (Optional) Train on halfcheetah too

In [ ]:
# Uncomment to also train on halfcheetah-medium-v2
# for agent_name, extra_args in agents:
#     cmd = [
#         "python", "train.py",
#         "--agent", agent_name,
#         "--dataset", "mujoco/halfcheetah/medium-v2",
#         "--steps", str(STEPS),
#         "--batch", str(BATCH),
#         "--device", DEVICE,
#         "--save_dir", "results/logs",
#     ]
#     subprocess.run(cmd)
print('Uncomment cell above to train on halfcheetah.')